# EarlyStruct – Design Explorer (Floors)
Carbon-first feasibility & comparison tool.

**How to use**
1) Set **Data dir** to the folder containing your CSVs.
2) (Optional) Provide a **Control file** path. If present, it takes precedence for project details and SPANS unless it contains `USE_CSV = Y`.
3) (Optional) Enter **Spans** like `9, 10.5` (meters) or `28ft, 32ft`. Notebook spans override control-file SPANS.
4) Click **Run Evaluation**. If no spans are provided (and no grid_options.csv / ideal spacing), the tool runs a 1-ft sweep (18–45 ft).


In [1]:
import sys
from pathlib import Path

import pandas as pd

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct import cli
from earlystruct.core import reporting


In [2]:
data_dir = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/csvs")
control_file = "/Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt"

df, ranked, pareto, saved = cli.evaluate(
    data_dir=data_dir,
    spans_str=None,          # let control file + sweep logic handle it
    export_dir=None,
    control_file=control_file,
)

if isinstance(saved, dict):
    project = saved.get("project", {}) or {}
    ctl = saved.get("ctl", {}) or {}
else:
    project = {}
    ctl = {}

df.shape, df.columns

((73100, 26),
 Index(['span_input_m', 'span_m', 'span_x_m', 'span_y_m', 'span_slab_dir_m',
        'span_beam_dir_m', 'system_id', 'system_name', 'category', 'type',
        'manufacturer', 'feasible', 'reason', 'depth_m', 'area_m2',
        'concrete_m3', 'steel_m3', 'timber_m3', 'carbon_total_kg',
        'carbon_per_m2', 'cost_total', 'cost_per_m2', 'nx', 'ny',
        'edge_canti_x_m', 'edge_canti_y_m'],
       dtype='object'))

In [ ]:
import os, sys
from pathlib import Path

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct.core.control import parse_control

control_path = "/Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt"

print("Exists?", os.path.exists(control_path), "->", control_path)

ctl = parse_control(control_path)
print("Parsed keys:", list(ctl.keys()))

print("PROJECT_NAME:", ctl.get("PROJECT_NAME"))
print("UNIT:", ctl.get("UNIT"))
print("FLOOR_AREA_PER_FLOOR:", ctl.get("FLOOR_AREA_PER_FLOOR"))
print("SPAN_SWEEP_FROM_MIN:", ctl.get("SPAN_SWEEP_FROM_MIN"), "BOOL:", ctl.get("SPAN_SWEEP_FROM_MIN_BOOL"))
print("ONE_WAY_IRREGULAR:", ctl.get("ONE_WAY_IRREGULAR"), "BOOL:", ctl.get("ONE_WAY_IRREGULAR_BOOL"))
print("ONE_WAY_SLAB_MIN_SPAN:", ctl.get("ONE_WAY_SLAB_MIN_SPAN"))
print("ONE_WAY_BEAM_MIN_SPAN:", ctl.get("ONE_WAY_BEAM_MIN_SPAN"))
print("PROGRAM_BLOCKS:", ctl.get("PROGRAM_BLOCKS"))


Exists? True -> /Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt
Parsed keys: ['USE_CSV', 'DATA_DIR', 'PROJECT_NAME', 'LOCATION', 'UNIT', 'FLOOR_AREA_PER_FLOOR', 'FLOOR_TO_FLOOR', 'NUM_FLOORS', 'IDEAL_COLUMN_SPACING', 'PLATE_LENGTH', 'PLATE_WIDTH', 'DEPTH_LIMIT_ENABLED', 'DEPTH_LIMIT', 'NOTES', 'SPANS', 'EDGE_CANTILEVER_MAX_M', 'SPAN_SWEEP_FROM_MIN', 'SPAN_SWEEP_STEP', 'LOAD_CHECK_MODE', 'RESULTS_SCOPE', 'ONE_WAY_IRREGULAR', 'ONE_WAY_SLAB_MIN_SPAN', 'ONE_WAY_BEAM_MIN_SPAN', 'USE_CSV_BOOL', 'DEPTH_LIMIT_ENABLED_BOOL', 'SPAN_SWEEP_FROM_MIN_BOOL', 'ONE_WAY_IRREGULAR_BOOL', 'PROGRAM_BLOCKS']
PROJECT_NAME: Warneford Research Building
UNIT: metric
FLOOR_AREA_PER_FLOOR: 5690.0
SPAN_SWEEP_FROM_MIN: Y BOOL: True
ONE_WAY_IRREGULAR: Y BOOL: True
ONE_WAY_SLAB_MIN_SPAN: 6.6
ONE_WAY_BEAM_MIN_SPAN: 9.0
PROGRAM_BLOCKS: [{'start_floor': 1, 'end_floor': 4, 'use': 'Laboratory_EU'}]


In [ ]:
UNIT = ctl.get("UNIT", "metric").lower()
is_imperial = UNIT.startswith("imp") or ("ft" in UNIT)

FT_PER_M = 3.28084

df_disp = df.copy()

if is_imperial:
    df_disp["span_slab_dir_ft"] = df_disp["span_slab_dir_m"] * FT_PER_M
    df_disp["span_beam_dir_ft"] = df_disp["span_beam_dir_m"] * FT_PER_M
else:
    df_disp["span_slab_dir_ft"] = pd.NA
    df_disp["span_beam_dir_ft"] = pd.NA

cols = [
    "system_id", "type", "feasible",
    "span_slab_dir_m", "span_beam_dir_m",
    "span_slab_dir_ft", "span_beam_dir_ft",
    "depth_m", "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_per_m2", "carbon_per_m2",
    "cost_total", "carbon_total_kg",
]

df_disp[[c for c in cols if c in df_disp.columns]].head(15)


,system_id,type,feasible,span_slab_dir_m,span_beam_dir_m,span_slab_dir_ft,span_beam_dir_ft,depth_m,area_m2,concrete_m3,steel_m3,timber_m3,cost_per_m2,carbon_per_m2,cost_total,carbon_total_kg
0,ComFlor100_normal_weight,composite_deck,False,8.0,10.909091,<NA>,<NA>,0.170900,22760.0,2188.753341,1050.6016,NaN,736.252000,769.888792,1.675710e+07,1.752267e+07
1,ComFlor210_normal_weight,composite_deck,False,8.0,10.909091,<NA>,<NA>,0.281250,22760.0,3756.348341,1803.0472,NaN,19.805000,57.632550,4.507618e+05,1.311717e+06
2,ComFlor80_normal_weight,composite_deck,False,8.0,10.909091,<NA>,<NA>,0.140900,22760.0,2188.753341,1050.6016,NaN,736.252000,769.888792,1.675710e+07,1.752267e+07
3,DuraStress_12DT24_2in_top,double_tee,False,8.0,10.909091,<NA>,<NA>,0.059267,22760.0,11772.413786,34140.0000,NaN,62.068965,180.620690,1.412690e+06,4.110927e+06
4,DuraStress_12DT24_no_top,double_tee,False,8.0,10.909091,<NA>,<NA>,0.055033,22760.0,7848.275857,22760.0000,NaN,41.379310,120.413793,9.417931e+05,2.740618e+06
5,DuraStress_12DT26_no_top,double_tee,False,8.0,10.909091,<NA>,<NA>,0.059275,22760.0,8319.172405,24125.6000,NaN,43.862069,127.638621,9.983007e+05,2.905055e+06
6,DuraStress_12DT28_2in_top,double_tee,False,8.0,10.909091,<NA>,<NA>,0.067725,22760.0,12557.241381,36416.0000,NaN,66.206897,192.662069,1.506869e+06,4.384989e+06
7,DuraStress_12DT28_no_top,double_tee,False,8.0,10.909091,<NA>,<NA>,0.063492,22760.0,8633.103452,25036.0000,NaN,45.517241,132.455172,1.035972e+06,3.014680e+06
8,DuraStress_12DT28_pretop,double_tee,False,8.0,10.909091,<NA>,<NA>,0.063508,22760.0,11458.482762,33229.6000,NaN,60.413793,175.804138,1.375018e+06,4.001302e+06
9,DuraStress_12DT30_pretop,double_tee,False,8.0,10.909091,<NA>,<NA>,0.067725,22760.0,11929.379310,34595.2000,NaN,62.896552,183.028966,1.431526e+06,4.165739e+06


In [ ]:
best_by_type = reporting.cheapest_span_by_type(df)

best_cols = [
    "type",
    "system_id",
    "span_slab_dir_m",
    "span_beam_dir_m",
    "depth_m",
    "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_total", "cost_per_m2",
    "carbon_total_kg", "carbon_per_m2",
]

best_by_type[[c for c in best_cols if c in best_by_type.columns]]


,type,system_id,span_slab_dir_m,span_beam_dir_m,depth_m,area_m2,concrete_m3,steel_m3,timber_m3,cost_total,cost_per_m2,carbon_total_kg,carbon_per_m2
0,beam_block,milbank_bb_155_md,8.0,10.909091,0.2550,22760.0,1688.033341,810.2560,NaN,2.025640e+05,8.900000,5.894612e+05,25.899000
1,clt_floor,klh_floor_5s_100mm_tl,8.0,10.909091,0.2000,22760.0,NaN,NaN,11152.400,7.249060e+06,318.500000,-5.338431e+06,-234.553200
2,composite_deck,spcf_nu_150_75_propped_LL0.75_S7.3,8.0,10.909091,0.2250,22760.0,3414.000000,1638.7200,NaN,4.096800e+05,18.000000,1.192169e+06,52.380000
3,double_tee,pci_12DT28,8.0,10.909091,0.8382,22760.0,2406.870000,1155.2976,NaN,2.888244e+05,12.690000,8.404790e+05,36.927900
4,hollowcore,bison_hc_150,8.0,10.909091,0.1500,22760.0,387.704833,1124.3440,NaN,6.978687e+04,3.066207,1.498091e+05,6.582124
5,lvl_joist,kerto_sbeam_joist_51x200_c400,8.0,10.909091,0.2000,22760.0,NaN,NaN,23215.200,2.089368e+07,918.000000,-9.923337e+06,-435.999000
6,lvl_panel,kerto_qpanel_floor_78mm,8.0,10.909091,0.1560,22760.0,NaN,NaN,9053.928,8.148535e+06,358.020000,-3.870102e+06,-170.039610
7,plank,flood_omnia_reinf,8.0,10.909091,0.1250,22760.0,1185.416659,569.0000,NaN,1.422500e+05,6.250000,4.139475e+05,18.187500
8,solid_plank,bison_sp_100_solid_plank,8.0,10.909091,0.1000,22760.0,2276.000000,1092.4800,NaN,2.731200e+05,12.000000,7.947792e+05,34.920000


In [3]:
reporting.print_verbose_summary(df, project, ctl)


NameError: name 'df' is not defined